In [1]:
import os
import pandas as pd
Dir_github = r'X:\PROJECTS\ABC@UofSC\3-Experiments\NSF_valence\github'
repo = 'nsf_valence'

In [34]:
Dir_ratings = r'X:\PROJECTS\ABC@UofSC\3-Experiments\NSF_valence\github\derivatives\behavioral\qualtrics'
df_ratings_raw = pd.read_csv(os.path.join(Dir_ratings,'df_scores_audio_att2_comp2.csv'))


df_ratings = df_ratings_raw.groupby(['audio','segID']).mean(numeric_only=True).reset_index()[['audio','segID','V','A']]
df_ratings['audio'] = df_ratings['audio'].str.replace('EmotionSentences_953','EmotionSentences')
df_N = df_ratings.groupby(['audio']).count()['segID']

Dir_timestamps = r'X:\PROJECTS\ABC@UofSC\3-Experiments\NSF_valence\github\derivatives\behavioral\timestamps\audacity_labels\doublechecked'
files = [x for x in os.listdir(Dir_timestamps) if '.txt' in x]
df_timestamps = []
for file in files:
    _,voice,story,_ = file.split('_')
    # if story == 'EmotionSentences':
    #     story = 'EmotionSentences_953'
    audio = f"{voice}_{story}"
    
    # check if the audio name of the timestamps file is the same as that in the rating file
    if audio not in df_ratings['audio'].unique():
        print(f"Data is not collected for {audio}")
    df = pd.read_csv(os.path.join(Dir_timestamps,file),header=None,sep='\t',names=['onset','offset','label'])

    # check if the number of segments match
    if df.shape[0] != df_N.loc[audio]:
        print(f"Number of segments do not match for {audio}")

    df['audio'] = audio
    df['segID'] = range(1,df.shape[0]+1)
    df_timestamps.append(df)
df_timestamps = pd.concat(df_timestamps,ignore_index=True)

df_ratings = pd.merge(left = df_ratings,left_on=['audio','segID'],
         right = df_timestamps,right_on=['audio','segID'],how='left')

C:\Users\xy6\AppData\Local\Temp\ipykernel_15208\314703309.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ratings_raw = pd.read_csv(os.path.join(Dir_ratings,'df_scores_audio_att2_comp2.csv'))


In [ ]:
Dir_segment = r'X:\PROJECTS\ABC@UofSC\3-Experiments\NSF_valence\github\derivatives\behavioral\timestamps\timestamps_segment'
df_segmentation = []
for audio in df_ratings['audio'].unique():
    path_file = os.path.join(Dir_segment,f"timestamps_{audio}_final.csv")
    if not os.path.exists(path_file):
        print(audio)
    df = pd.read_csv(path_file)

    # check if the number of segments match
    if df.shape[0] != df_N.loc[audio]:
        print(f"Number of segments do not match for {audio}")
    df['segID'] = df['idx']+1

    df_segmentation.append(df)
df_segmentation = pd.concat(df_segmentation,ignore_index=True)
df_segmentation['audio'] = df_segmentation['audio'].str.replace('BradfordBoar','BradfordBull')

df_ratings = pd.merge(left = df_ratings,left_on=['audio','segID'],
         right = df_segmentation[['segment','reason','story','audio','words','segID']],right_on=['audio','segID'],how='left')

Number of segments do not match for Jacob_UserFriendly


In [ ]:
df_ratings.to_csv(os.path.join(Dir_github,repo,'derivatives','behavioral','ratings','audio','ratings_audio_att2_comp2_final.csv'),index=False)